In [8]:
import sys
from pydantic import BaseModel
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage
from langgraph.graph import StateGraph, START, END
from typing import Annotated
from operator import add
from dotenv import load_dotenv

load_dotenv()


class AgentState(BaseModel):
    user_input: str = ""
    category: str = ""
    thinking: Annotated[list[str], add] = []
    response: str = ""


class UserContext(BaseModel):
    user_name: str = "Carlos"
    tone: str = "casual"
    language: str = "pt-br"


context = UserContext(user_name="Carlos", tone="casual", language="pt-br")

llm = ChatOpenAI(model="gpt-4.1-mini", temperature=0)


def classify_node(state: AgentState) -> dict:
    result = llm.invoke([
        SystemMessage(content="Classifique a pergunta em UMA palavra: tecnologia, saude, financas ou geral."),
        HumanMessage(content=state.user_input),
    ])
    category = result.content.strip().lower()
    print(f"→ Categoria: '{category}'", flush=True)

    return {
        "category": category,
        "thinking": [f"🏷️ Classificação: '{category}'"],
    }


def reason_node(state: AgentState) -> dict:
    print("→ Gerando raciocínio passo a passo...", flush=True)
    full_content = ""
    for chunk in llm.stream([
        SystemMessage(content=(
            f"Você é um assistente especializado em {state.category}.\n"
            f"Contexto do usuário: {context.model_dump()}\n\n"
            "Antes de responder, pense passo a passo sobre a pergunta. "
            "Crie uma lista de etapas ou pontos chave importantes que você tem que fazer/executar/obter/elaborar para responder a pergunta."
            "Escreva APENAS seu raciocínio interno, de forma resumida e objetiva, não a resposta final."
        )),
        HumanMessage(content=state.user_input),
    ]):
        token = chunk.content
        print(token, end="", flush=True)
        full_content += token
    print()
    return {
        "thinking": [f"🧠 Raciocínio:\n{full_content}"],
    }


def respond_node(state: AgentState) -> dict:
    print("→ Gerando resposta...", flush=True)
    reasoning = "\n".join(state.thinking)
    full_response = ""
    for chunk in llm.stream([
        SystemMessage(content=(
            f"Você é um assistente com tom {context.tone} para {context.user_name}.\n"
            f"Categoria: {state.category}\n\n"
            f"Seu raciocínio prévio:\n{reasoning}\n\n"
            "Agora escreva a resposta final para o usuário, "
            "clara e bem estruturada, baseada no seu raciocínio."
        )),
        HumanMessage(content=state.user_input),
    ]):
        token = chunk.content
        print(token, end="", flush=True)
        full_response += token

    print()
    return {
        "response": full_response,
        "thinking": ["✅ Resposta gerada com sucesso"],
    }



graph = StateGraph(AgentState)

graph.add_node("classify", classify_node)
graph.add_node("reason", reason_node)
graph.add_node("respond", respond_node)

graph.add_edge(START, "classify")
graph.add_edge("classify", "reason")
graph.add_edge("reason", "respond")
graph.add_edge("respond", END)

app = graph.compile()


result = app.invoke({
    "user_input": "Quais são as melhores práticas para dormir melhor?"
})


→ Categoria: 'saude'
→ Gerando raciocínio passo a passo...
1. Identificar fatores que influenciam a qualidade do sono, como ambiente, rotina e hábitos.  
2. Listar práticas recomendadas para melhorar o sono, como manter horário regular, evitar eletrônicos antes de dormir, controlar a alimentação e a ingestão de cafeína.  
3. Incluir dicas sobre ambiente adequado para dormir, como temperatura, iluminação e conforto do colchão.  
4. Mencionar a importância de atividades relaxantes antes de dormir, como leitura ou meditação.  
5. Considerar a necessidade de evitar cochilos longos durante o dia.  
6. Ressaltar a importância de exercícios físicos regulares, mas não próximos da hora de dormir.  
7. Abordar a possível necessidade de consultar um especialista em casos de insônia persistente.
→ Gerando resposta...
Oi Carlos! Dormir bem faz toda a diferença, né? Aqui vão algumas dicas que podem ajudar você a melhorar a qualidade do seu sono:

1. **Mantenha um horário regular:** Tente dormir e ac

In [ ]:
import sys
import os
sys.path.insert(0, os.path.join(os.path.dirname(__file__), ".."))

from dotenv import load_dotenv
load_dotenv()

from deep_agent import DeepAgent, UserContext

context = UserContext(user_name="Carlos", tone="casual", language="pt-br")

agent = DeepAgent(
    model="gpt-4.1-mini",
    user_context=context,
    streaming=True,
)

result = agent.invoke("Quais são as melhores práticas para dormir melhor?")

print("\n💬 RESPOSTA FINAL")
print("─" * 55)
print(result["response"])
